# Homework 6

Target is fuel_efficiency_mpg. Missing values filled with 0, split 60/20/20 with random_state=1.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(url).fillna(0)

In [2]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

y_train = df_train.pop('fuel_efficiency_mpg').values
y_val = df_val.pop('fuel_efficiency_mpg').values

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
len(y_train), len(y_val)

(6000, 2000)

## Q1 - feature used by a depth 1 tree

In [3]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



**Q1: `'model_year'`**, split at 1997.5. Older cars average 28.6 mpg, newer ones 31.2.


## Q2 - random forest, 10 trees

In [4]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
round(root_mean_squared_error(y_val, rf.predict(X_val)), 4)

1.8332

**Q2: 1.837.** My run gives 1.833.

The course's own `validate_homework.py`, run here with the same calls, also gives 1.8332,
so the gap comes from the scikit-learn version (1.9.0 here), not the split. The other
options are a factor of 10 away, so there is no doubt about which one it is.


## Q3 - n_estimators

In [5]:
scores = {}
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    scores[n] = round(root_mean_squared_error(y_val, rf.predict(X_val)), 3)
scores

{10: 1.833, 50: 1.774, 100: 1.77, 150: 1.769}

**Q3: 150**, 1.769. The curve is almost flat after 50: 1.774, 1.770, 1.769.


## Q4 - max_depth by mean RMSE

In [6]:
means = {}
for d in [10, 15, 20, 25]:
    s = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        s.append(root_mean_squared_error(y_val, rf.predict(X_val)))
    means[d] = round(float(np.mean(s)), 4)
means

{10: 1.7544, 15: 1.7842, 20: 1.7868, 25: 1.7863}

**Q4: 10**, mean RMSE 1.754. Deeper trees do worse, 15 to 25 all sit around 1.785.


## Q5 - feature importance

In [7]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
imp = pd.Series(rf.feature_importances_, index=dv.get_feature_names_out())
imp[['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']].sort_values(ascending=False)

vehicle_weight         0.203313
horsepower             0.078271
engine_displacement    0.061281
acceleration           0.054998
dtype: float64

**Q5: `vehicle_weight`**, 0.203, more than double `horsepower` at 0.078.


## Q6 - xgboost eta

In [8]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, 'train'), (dval, 'val')]

res = {}
for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,

        'objective': 'reg:squarederror',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }
    model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    res[eta] = round(root_mean_squared_error(y_val, model.predict(dval)), 4)
res

{0.3: 1.8265, 0.1: 1.7248}

**Q6: 0.1**, RMSE 1.725 against 1.827 for 0.3.

With 100 rounds and eta 0.3 the model overfits the training set; 0.1 learns slower and
stops in a better place.
